<a href="https://colab.research.google.com/github/stevenolanecon/7002LBSAI/blob/main/notebooks/week5_inclass_tasks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Before you start:**

- Go to **File → Save a copy in Drive**. Until you do this, your work only exists in this browser tab – you can't save changes back to GitHub, and closing the tab or losing the session will lose your work.
- Turn off Colab's autocomplete: **Tools → Settings → Editor → uncheck "Show context-powered code completions."** It's a setting on your own Google account, not something built into this notebook, so you'll need to do it once per account.

# Week 5 – In-class tasks

Seven short tasks on our own class data, following the order of the workshop slides. Each one builds on Week 4's model, `mean_rt ~ finger_mm`. Where a task needs syntax you haven't seen yet, there's a hint with generic names (`df`, `x`, `y`) – swap in the real ones.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Our class data from Week 1, with the finger-length typos fixed in Week 3.
data_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/week3_reaction_times.csv"
df = pd.read_csv(data_path)
df['finger_mm'] = df['finger_length_corrected']

# One row per student, as in Week 4: a student's trials aren't independent.
student_summary = (
    df.groupby('name')
    .agg(mean_rt=('reaction_time_ms', 'mean'), finger_mm=('finger_mm', 'first'),
         device=('device', 'first'))
    .reset_index()
)

# Week 4's model, which most tasks build on
model = smf.ols('mean_rt ~ finger_mm', data=student_summary).fit()
model.params

## Task 1 – log our own outcome

**Business question:** reaction time is right-skewed. Does a log model tell a different story?

Fit a **log–level** model, `np.log(mean_rt) ~ finger_mm`. You can take the log inside the formula:

```python
model_log = smf.ols('np.log(y) ~ x', data=df).fit()
```

Interpret the slope as a percentage, and compare R² with Week 4's model (`model.rsquared`).

## Task 2 – fit a quadratic

**Business question:** is the relationship curved?

Add a squared term with `I(...)`:

```python
model_quad = smf.ols('y ~ x + I(x**2)', data=df).fit()
```

Compare R² with the straight line. Then work out the slope at 80mm: $\hat\beta_1 + 2\hat\beta_2 \times 80$.

## Task 3 – one influential student?

**Business question:** is the slope driven by one person?

Plot or look at the data, pick the student you think matters most, and refit Week 4's model **without** them. To keep every row except one:

```python
df_without = df[df['name'] != 'Student 1']
```

How much does the slope change? What would you do with that student, and why?

## Task 4 – QQ-plot of our own residuals

**Business question:** are our residuals close enough to normal for the CI to be trusted?

Make a QQ-plot of the residuals from Week 4's model, then do the same for the log model from Task 1:

```python
import statsmodels.api as sm
sm.qqplot(model.resid, line='s')
plt.show()
```

Which looks more like a straight line? Check `Prob(JB)` and `Skew` in each model's `.summary()` too.

## Task 5 – classical vs. robust SE

**Business question:** does allowing for heteroskedasticity change how sure we are about the slope?

Fit Week 4's model again with robust standard errors:

```python
model_robust = smf.ols('y ~ x', data=df).fit(cov_type='HC1')
```

Compare `model.params`, `model.bse` and `model.conf_int()` with the same three for `model_robust`. Which numbers change, and which don't?

## Task 6 – is finger length significant?

**Business question:** can HR rely on this slope at all?

Print the full summary with `print(model.summary())` and read off the t-statistic and `P>|t|` for `finger_mm`. Can you reject $H_0: \beta = 0$ at the 5% level? Does the CI agree? You can also pull them out directly with `model.tvalues` and `model.pvalues`.

## Task 7 – CI vs. prediction interval at 80mm

**Business question:** what can HR say about one new candidate with an 80mm finger?

```python
new = pd.DataFrame({'x': [value]})
model.get_prediction(new).summary_frame(alpha=0.05)
```

`mean_ci_lower` / `mean_ci_upper` is the confidence interval for the **average** student at 80mm; `obs_ci_lower` / `obs_ci_upper` is the prediction interval for **one** student. How much wider is the second, and why?

## Extra – both bands on one chart

This is the chart from the **Both bands, on our own data** slide. `get_prediction()` is the same call as Task 7, but given 100 finger lengths instead of one. `mean_ci_lower`/`mean_ci_upper` is the confidence interval for the average student (red); `obs_ci_lower`/`obs_ci_upper` is the prediction interval for one student (blue).

The bands are very slightly wider than on the slide: the slide uses ±2 × SE, statsmodels uses the exact t critical value (about 2.07 with 24 students).

In [ ]:
model = smf.ols('mean_rt ~ finger_mm', data=student_summary).fit()

# a grid of finger lengths across the range of the data
grid = pd.DataFrame({'finger_mm': np.linspace(student_summary['finger_mm'].min(),
                                              student_summary['finger_mm'].max(), 100)})

# both intervals at every point on the grid
bands = model.get_prediction(grid).summary_frame(alpha=0.05)

fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(grid['finger_mm'], bands['obs_ci_lower'], bands['obs_ci_upper'],
                color='#457b9d', alpha=0.15, label='Prediction interval (one student)')
ax.fill_between(grid['finger_mm'], bands['mean_ci_lower'], bands['mean_ci_upper'],
                color='#e63946', alpha=0.3, label='CI for the line (average student)')
ax.plot(grid['finger_mm'], bands['mean'], color='black', label='Fitted line')
ax.scatter(student_summary['finger_mm'], student_summary['mean_rt'], color='black', s=20)
ax.set(xlabel='Finger length (mm)', ylabel='Mean reaction time (ms)')
ax.legend()
plt.show()